# **SelectorGroupChat-Supervisor Agent**

![alt text](supervisor.png "Selector Group Chat Agent")

## **This system uses three specialized agents:**

- **Planning Agent:** The strategic coordinator that breaks down complex tasks into manageable subtasks.

- **Web Search Agent:** An information retrieval specialist that interfaces with the `search_web_tool`.

- **Data Analyst Agent:** An agent specialist in performing calculations equipped with percentage_change_tool.

The tools `search_web_tool` and `percentage_change_tool` are external tools that the agents can use to perform their tasks.

In [1]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
import asyncio

## **Ollama Model**

In [2]:
ollama_model_clint = OllamaChatCompletionClient(model="llama3.1")

In [3]:
from typing import List, Sequence

from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.ui import Console


In [4]:
# Note: This example uses mock tools instead of real APIs for demonstration purposes
def search_web_tool(query: str) -> str:
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."


def percentage_change_tool(start: float, end: float) -> float:
    return ((end - start) / start) * 100


In [5]:
model_client = ollama_model_clint

planning_agent = AssistantAgent(
    "PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

web_search_agent = AssistantAgent(
    "WebSearchAgent",
    description="An agent for searching information on the web.",
    tools=[search_web_tool],
    model_client=model_client,
    system_message="""
    You are a web search agent.
    Your only tool is search_tool - use it to find information.
    You make only one search call at a time.
    Once you have the results, you never do calculations based on them.
    """,
)

data_analyst_agent = AssistantAgent(
    "DataAnalystAgent",
    description="An agent for performing calculations.",
    model_client=model_client,
    tools=[percentage_change_tool],
    system_message="""
    You are a data analyst.
    Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided.
    If you have not seen the data, ask for it.
    """,
)


In [6]:
text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=25)
termination = text_mention_termination | max_messages_termination


In [7]:
selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""


In [8]:
team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,  # Allow an agent to speak multiple turns in a row.
)


In [9]:
task = "Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [10]:
# Use asyncio.run(...) if you are running this in a script.
await Console(team.run_stream(task=task))


---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To find out who was the Miami Heat player with the highest points in the 2006-2007 season, I'll assign tasks to my team members:

1. WebSearchAgent : Search for the top scorer of the Miami Heat during the 2006-2007 NBA season.

Once this information is obtained, I'll then ask DataAnalystAgent to calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons for that player:

1. WebSearchAgent : Retrieve the Miami Heat's roster and box scores for the 2007-2008 and 2008-2009 seasons.

2. DataAnalystAgent : Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons for the player identified by WebSearchAgent as the top scorer of the Miami Heat during the 20

TaskResult(messages=[TextMessage(id='f4c7dd45-a551-47c1-8302-98a53790cd25', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 21, 9, 12, 5, 645793, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='01c8dd6f-0c4a-4a71-8ed5-6143ff98ff69', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=163, completion_tokens=206), metadata={}, created_at=datetime.datetime(2025, 7, 21, 9, 12, 38, 328996, tzinfo=datetime.timezone.utc), content="To find out who was the Miami Heat player with the highest points in the 2006-2007 season, I'll assign tasks to my team members:\n\n1. WebSearchAgent : Search for the top scorer of the Miami Heat during the 2006-2007 NBA season.\n\nOnce this information is obtained, I'll then ask DataAnalystAgent to calculate 

## **Custom Selector Function**

Often times we want better control over the selection process. To this end, we can set the selector_func argument with a custom selector function to override the default model-based selection. This allows us to implement more complex selection logic and state-based transitions.

For instance, we want the Planning Agent to speak immediately after any specialized agent to check the progress.

In [12]:
def selector_func(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None:
    if messages[-1].source != planning_agent.name:
        return planning_agent.name
    return None


# Reset the previous team and run the chat again with the selector function.
await team.reset()
team = SelectorGroupChat(
    [planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
    selector_func=selector_func,
)

await Console(team.run_stream(task=task))


---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
1. WebSearchAgent : Search for the top scorers of the 2006-2007 Miami Heat team.
2. DataAnalystAgent : Analyze the rebound statistics of the players from the previous task, focusing on a specific player's total rebounds in the 2007-2008 and 2008-2009 seasons.

Please wait for the agents to complete their tasks.
---------- TextMessage (WebSearchAgent) ----------
To answer the question about the Miami Heat player with the highest points in the 2006-2007 season and the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons, I will call the search_web_tool function.

{"name": "search_web_tool", "parameters": {"query": "Miami Heat top scorer 2006-2007 season"}}
---------- TextMessage (PlanningAg

TaskResult(messages=[TextMessage(id='cad53dd3-d03b-4238-90cf-a1b022e45baa', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 21, 10, 43, 24, 427506, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest points in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='8d13ca5c-689c-4184-8adb-cd595162c7cb', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=163, completion_tokens=80), metadata={}, created_at=datetime.datetime(2025, 7, 21, 10, 43, 29, 725968, tzinfo=datetime.timezone.utc), content="1. WebSearchAgent : Search for the top scorers of the 2006-2007 Miami Heat team.\n2. DataAnalystAgent : Analyze the rebound statistics of the players from the previous task, focusing on a specific player's total rebounds in the 2007-2008 and 2008-2009 seasons.\n\nPlease wait for the agents to complete their 